In [65]:
%matplotlib inline
import torch,os,torchvision
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, models, transforms
from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit
torch.__version__

'2.9.1'

4.1 Fine tuning 模型微调
在前面的介绍卷积神经网络的时候，说到过PyTorch已经为我们训练好了一些经典的网络模型，那么这些预训练好的模型是用来做什么的呢？其实就是为了我们进行微调使用的。

4.1.1 什么是微调

针对于某个任务，自己的训练数据不多，那怎么办？
没关系，我们先找到一个同类的别人训练好的模型，把别人现成的训练好了的模型拿过来，换成自己的数据，调整一下参数，再训练一遍，这就是微调（fine-tune）。
PyTorch里面提供的经典的网络模型都是官方通过Imagenet的数据集与训练好的数据，如果我们的数据训练数据不够，这些数据是可以作为基础模型来使用的。

为什么要微调
1. 对于数据集本身很小（几千张图片）的情况，从头开始训练具有几千万参数的大型神经网络是不现实的，因为越大的模型对数据量的要求越大，过拟合无法避免。这时候如果还想用上大型神经网络的超强特征提取能力，只能靠微调已经训练好的模型。
2. 可以降低训练成本：如果使用导出特征向量的方法进行迁移学习，后期的训练成本非常低，用 CPU 都完全无压力，没有深度学习机器也可以做。
3. 前人花很大精力训练出来的模型在大概率上会比你自己从零开始搭的模型要强悍，没有必要重复造轮子。


迁移学习 Transfer Learning
总是有人把 迁移学习和神经网络的训练联系起来，这两个概念刚开始是无关的。
迁移学习是机器学习的分支，现在之所以 迁移学习和神经网络联系如此紧密，现在图像识别这块发展的太快效果也太好了，所以几乎所有的迁移学习都是图像识别方向的，所以大家看到的迁移学习基本上都是以神经网络相关的计算机视觉为主，本文中也会以这方面来举例子

迁移学习初衷是节省人工标注样本的时间，让模型可以通过一个已有的标记数据的领域向未标记数据领域进行迁移从而训练出适用于该领域的模型，直接对目标域从头开始学习成本太高，我们故而转向运用已有的相关知识来辅助尽快地学习新知识

举一个简单的例子就能很好的说明问题，我们学习编程的时候会学习什么？ 语法、特定语言的API、流程处理、面向对象，设计模式，等等

这里面语法和API是每一个语言特有的，但是面向对象和设计模式可是通用的，我们学了JAVA，再去学C#，或者Python，面向对象和设计模式是不用去学的，因为原理都是一样的，甚至在学习C#的时候语法都可以少学很多，这就是迁移学习的概念，把统一的概念抽象出来，只学习不同的内容。

迁移学习按照学习方式可以分为基于样本的迁移，基于特征的迁移，基于模型的迁移，以及基于关系的迁移，这里就不详细介绍了。

二者关系
其实 "Transfer Learning" 和 "Fine-tune" 并没有严格的区分，含义可以相互交换，只不过后者似乎更常用于形容迁移学习的后期微调中。
我个人的理解，微调应该是迁移学习中的一部分。微调只能说是一个trick。

4.1.2 如何微调
对于不同的领域微调的方法也不一样，比如语音识别领域一般微调前几层，图片识别问题微调后面几层，这个原因我这里也只能讲个大概，具体还要大神来解释：

对于图片来说，我们CNN的前几层学习到的都是低级的特征，比如，点、线、面，这些低级的特征对于任何图片来说都是可以抽象出来的，所以我们将他作为通用数据，只微调这些低级特征组合起来的高级特征即可，例如，这些点、线、面，组成的是圆还是椭圆，还是正方形，这些代表的含义是我们需要后面训练出来的。

对于语音来说，每个单词表达的意思都是一样的，只不过发音或者是单词的拼写不一样，比如 苹果，apple，apfel（德语），都表示的是同一个东西，只不过发音和单词不一样，但是他具体代表的含义是一样的，就是高级特征是相同的，所以我们只要微调低级的特征就可以了。

下面只介绍下计算机视觉方向的微调，摘自 [cs231](http://cs231n.github.io/transfer-learning/)

 - ConvNet as fixed feature extractor.：
其实这里有两种做法： 
1. 使用最后一个fc layer之前的fc layer获得的特征，学习个线性分类器(比如SVM) 
2. 重新训练最后一个fc layer


 - Fine-tuning the ConvNet
 
固定前几层的参数，只对最后几层进行fine-tuning,
 
对于上面两种方案有一些微调的小技巧，比如先计算出预训练模型的卷积层对所有训练和测试数据的特征向量，然后抛开预训练模型，只训练自己定制的简配版全连接网络。
这个方式的一个好处就是节省计算资源，每次迭代都不会再去跑全部的数据，而只是跑一下简配的全连接

 
 - Pretrained models 
 
这个其实和第二种是一个意思，不过比较极端，使用整个pre-trained的model作为初始化，然后fine-tuning整个网络而不是某些层，但是这个的计算量是非常大的,就只相当于做了一个初始化。




4.1.3 注意事项

1. 新数据集和原始数据集合类似，那么直接可以微调一个最后的FC层或者重新指定一个新的分类器
2. 新数据集比较小和原始数据集合差异性比较大，那么可以使用从模型的中部开始训练，只对最后几层进行fine-tuning
3. 新数据集比较小和原始数据集合差异性比较大，如果上面方法还是不行的化那么最好是重新训练，只将预训练的模型作为一个新模型初始化的数据
4. 新数据集的大小一定要与原始数据集相同，比如CNN中输入的图片大小一定要相同，才不会报错
5. 如果数据集大小不同的话，可以在最后的fc层之前添加卷积或者pool层，使得最后的输出与fc层一致，但这样会导致准确度大幅下降，所以不建议这样做
6. 对于不同的层可以设置不同的学习率，一般情况下建议，对于使用的原始数据做初始化的层设置的学习率要小于（一般可设置小于10倍）初始化的学习率，这样保证对于已经初始化的数据不会扭曲的过快，而使用初始化学习率的新层可以快速的收敛。

4.1.3 微调实例
这里面我们使用官方训练好的resnet50来参加kaggle上面的 [dog breed](https://www.kaggle.com/c/dog-breed-identification) 狗的种类识别来做一个简单微调实例。

首先我们需要下载官方的数据解压，只要保持数据的目录结构即可，这里指定一下目录的位置，并且看下内容
手动下载 https://www.kaggle.com/competitions/dog-breed-identification/data

In [66]:
DATA_ROOT = 'data'
all_labels_df = pd.read_csv(os.path.join(DATA_ROOT,'labels.csv'))
print(type(all_labels_df))  # <class 'pandas.core.frame.DataFrame'>
all_labels_df.head()

<class 'pandas.core.frame.DataFrame'>


,id,breed
0,000bec180eb18c7604dcecc8fe0dba07,boston_bull
1,001513dfcb2ffafc82cccf4d8bbaba97,dingo
2,001cdf01b096e06d78e9e5112d419397,pekinese
3,00214f311d5d2247d5dfe4fe24b2303d,bluetick
4,0021f9ceb3235effd7fcde7f7538ed62,golden_retriever


获取狗的分类根据分类进行编号

这里定义了两个字典，分别以名字和id作为对应，方便后面处理

In [67]:
breeds = all_labels_df.breed.unique()
print(type(breeds),breeds.shape) # <class 'numpy.ndarray'>   (120,)
breed2idx = dict((breed,idx) for idx,breed in enumerate(breeds))
# print(breed2idx)
idx2breed = dict((idx,breed) for idx,breed in enumerate(breeds))
len(breeds)

<class 'numpy.ndarray'> (120,)


120

添加到列表中

In [68]:
all_labels_df['label_idx'] = [breed2idx[b] for b in all_labels_df.breed]
all_labels_df.head()

,id,breed,label_idx
0,000bec180eb18c7604dcecc8fe0dba07,boston_bull,0
1,001513dfcb2ffafc82cccf4d8bbaba97,dingo,1
2,001cdf01b096e06d78e9e5112d419397,pekinese,2
3,00214f311d5d2247d5dfe4fe24b2303d,bluetick,3
4,0021f9ceb3235effd7fcde7f7538ed62,golden_retriever,4


由于我们的数据集不是官方指定的格式，我们自己定义一个数据集

In [69]:
class DogDataset(Dataset):
    def __init__(self, labels_df, img_path, transform=None):
        self.labels_df = labels_df
        self.img_path = img_path
        self.transform = transform
        
    def __len__(self):
        return self.labels_df.shape[0]
    
    def __getitem__(self, idx):
        image_name = os.path.join(self.img_path, self.labels_df.id[idx]) + '.jpg'
        # 输出 image_name: data/train/26914931b40a466c795993e6abd78673.jpg
        # print('image_name:', image_name)
        img = Image.open(image_name)
        label = self.labels_df.label_idx[idx]
        # 输出 label: 118
        # print('label:', label)
        if self.transform:
            img = self.transform(img)
        return img, label

定义一些超参数

In [70]:
# 224 不是随便选的数字，主要原因是：
# 经典预训练模型（如 ResNet、VGG、AlexNet）的设计输入尺寸就是 224×224，使用这个尺寸能直接复用这些模型的预训练权重，无需额外调整网络结构；
# 224×224 在计算效率和模型性能之间达到了较好的平衡（尺寸太小会丢失细节，太大则会增加计算量）。
IMG_SIZE = 224 # resnet50的输入是224的所以需要将图片统一大小
BATCH_SIZE= 256 #这个批次大小需要占用4.6-5g的显存，如果不够的化可以改下批次，如果内存超过10G可以改为512
IMG_MEAN = [0.485, 0.456, 0.406]
IMG_STD = [0.229, 0.224, 0.225]
CUDA=torch.cuda.is_available()
# DEVICE = torch.device("cuda" if CUDA else "cpu")
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = torch.device('mps')  # Apple Silicon (M1/M2/M3)
else:
    DEVICE = torch.device('cpu')

定义训练和验证数据的图片变换规则

In [71]:
# 图像变换的核心：训练集随机增强（抗过拟合），验证集固定标准化（保准确）；
# 随机变换（RandomXXX）：增加数据多样性，让模型学通用特征；
# 标准化（ToTensor+Normalize）：适配模型输入格式，加速收敛
train_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),  # 把图像的短边缩放到 224 像素（长边按比例缩放）
    transforms.RandomResizedCrop(IMG_SIZE),  # 随机裁剪出目标尺寸的区域（模拟不同视角）
    transforms.RandomHorizontalFlip(),  # 随机水平翻转（50%概率）
    transforms.RandomRotation(30),  # 随机旋转±30度
    transforms.ToTensor(),  # 将PIL图像/NumPy数组转为Tensor，同时把像素值从[0,255]缩放到[0,1]
    transforms.Normalize(IMG_MEAN, IMG_STD)  # 按均值和标准差归一化，加速模型收敛
])

val_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),  # 缩放到目标尺寸（保持长宽比）
    transforms.CenterCrop(IMG_SIZE),  # 就是从图像中心裁剪出 224×224 的正方形区域。
    transforms.ToTensor(),  # 转为Tensor并缩放像素值
    transforms.Normalize(IMG_MEAN, IMG_STD)  # 与训练集使用相同的归一化参数（关键！）
])

我们这里只分割10%的数据作为训练时的验证数据

In [72]:
dataset_names = ['train', 'valid']
# 普通的随机划分可能导致验证集中某些品种的样本过少甚至没有，分层划分能保证训练集和验证集中各品种的比例与原数据集一致，尤其适合样本分布不均衡的分类任务（比如宠物品种分类）。
# 初始化分层随机划分器
# n_splits=1：只划分1次
# test_size=0.1：验证集占10%，训练集占90%
# random_state=0：固定随机种子，保证每次运行划分结果一致
stratified_split = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=0)
# 执行划分，获取训练/验证集的索引
# split(X, y)：y必须传入分类标签（这里是all_labels_df.breed），X可以是任意维度的特征（这里用id列仅作为样本标识）。
# next(iter(...))：取出唯一的一次划分结果
train_split_idx, val_split_idx = next(iter(stratified_split.split(all_labels_df.id, all_labels_df.breed)))
# 根据索引提取训练集/验证集，并重置索引（避免原索引混乱）
print('type(all_labels_df)=',type(all_labels_df)) # <class 'pandas.core.frame.DataFrame'>
# reset_index()，但默认会保留原索引为新列（列名是index），建议改为reset_index(drop=True)，避免数据框中多出无用的列。
train_df = all_labels_df.iloc[train_split_idx].reset_index()
print('type(train_df)=',type(train_df)) # <class 'pandas.core.frame.DataFrame'>
val_df = all_labels_df.iloc[val_split_idx].reset_index()
# train_df.head()
print(len(train_df))
print(len(val_df))
train_df.head()

type(all_labels_df)= <class 'pandas.core.frame.DataFrame'>
type(train_df)= <class 'pandas.core.frame.DataFrame'>
9199
1023


,index,id,breed,label_idx
0,9556,efbabde6fc97bb48c8c8b6b75bfaea59,eskimo_dog,78
1,2055,332c413119b474653ecca0f358c85e1f,giant_schnauzer,29
2,5652,8e7256b23446acbd33967122787c1eb3,tibetan_mastiff,116
3,7462,bb7fdde5ce18544f51b1091f8f14533f,labrador_retriever,25
4,9279,e8f5dd1ad67209c064965691030a07e5,weimaraner,28


使用官方的dataloader载入数据

In [73]:
image_transforms = {'train':train_transforms, 'valid':val_transforms}

train_dataset = DogDataset(train_df, os.path.join(DATA_ROOT,'train'), transform=image_transforms['train'])
val_dataset = DogDataset(val_df, os.path.join(DATA_ROOT,'train'), transform=image_transforms['valid'])
image_dataset = {'train':train_dataset, 'valid':val_dataset}
# 一次性为训练集和验证集分别创建对应的 DataLoader 对象，并封装到字典中统一管理
# num_workers=0：使用主进程加载数据（单进程），这是所有系统的默认值
image_dataloader = {x:DataLoader(image_dataset[x],batch_size=BATCH_SIZE,shuffle=True,num_workers=0) for x in dataset_names}
dataset_sizes = {x:len(image_dataset[x]) for x in dataset_names}

开始配置网络，由于ImageNet是识别1000个物体，我们的狗的分类一共只有120，所以需要对模型的最后一层全连接层进行微调，将输出从1000改为120

In [74]:
from torchvision.models import ResNet50_Weights

model_ft = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1) # 这里自动下载官方的预训练模型，并且
# 将所有的参数层进行冻结
for param in model_ft.parameters():
    param.requires_grad = False
# 这里打印下全连接层的信息
print(model_ft.fc)
num_fc_ftr = model_ft.fc.in_features #获取到fc层的输入
model_ft.fc = nn.Linear(num_fc_ftr, len(breeds)) # 定义一个新的FC层
model_ft=model_ft.to(DEVICE)# 放到设备中
# print(model_ft) # 打印一下新的模型
print(model_ft.fc) # 这里打印下新的全连接层的信息

Linear(in_features=2048, out_features=1000, bias=True)
Linear(in_features=2048, out_features=120, bias=True)


设置训练参数

In [75]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam([
    {'params':model_ft.fc.parameters()}
], lr=0.001)#指定 新加的fc层的学习率

定义训练函数

In [76]:
def train(model,device, train_loader, epoch):
    model.train()
    for batch_idx, data in enumerate(train_loader):
        x,y= data
        x=x.to(device)
        y=y.to(device)
        optimizer.zero_grad()
        y_hat= model(x)
        loss = criterion(y_hat, y)
        loss.backward()
        optimizer.step()
    print ('Train Epoch: {}\t Loss: {:.6f}'.format(epoch,loss.item()))

定义测试函数

In [77]:
def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for i,data in enumerate(test_loader):          
            x,y= data
            x=x.to(device)
            y=y.to(device)
            optimizer.zero_grad()
            y_hat = model(x)
            test_loss += criterion(y_hat, y).item() # sum up batch loss
            pred = y_hat.max(dim=1, keepdim=True)[1] # get the index of the max log-probability

            # type(y_hat)= <class 'torch.Tensor'> torch.Size([255, 120])
            # type(pred)= <class 'torch.Tensor'> torch.Size([255, 1])
            # type(y)= <class 'torch.Tensor'> torch.Size([255])
            # type(y_hat.max(1, keepdim=True))= <class 'torch.return_types.max'>
            print('type(y_hat)=',type(y_hat),y_hat.shape)
            print('type(pred)=',type(pred),pred.shape)
            print('type(y)=',type(y),y.shape)
            print('type(y_hat.max(1, keepdim=True))=',type(y_hat.max(1, keepdim=True)))

            correct += pred.eq(y.view_as(pred)).sum().item()
    test_loss /= len(test_loader.dataset)
    # Test set: Average loss: 0.0189, Accuracy: 15/1023 (1%)
    print('\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
        test_loss, correct, len(val_dataset),
        100. * correct / len(val_dataset)))

训练9次，看看效果

In [78]:
# RuntimeError: Input type (MPSFloatType) and weight type (torch.FloatTensor) should be the same 错误，核心原因是模型参数（权重）和输入数据的设备 / 数据类型不匹配—— 你的输入数据被放到了苹果 MPS 设备（Apple Silicon 的 GPU）上，但模型权重还留在 CPU 上，导致计算时类型冲突。
# for epoch in range(1, 10):
print('DEVICE=',DEVICE)
for epoch in range(1, 2):
    %time train(model=model_ft,device=DEVICE, train_loader=image_dataloader["train"],epoch=epoch)
    test(model=model_ft, device=DEVICE, test_loader=image_dataloader["valid"])

DEVICE= mps
Train Epoch: 1	 Loss: 2.839274
CPU times: user 14.3 s, sys: 8.85 s, total: 23.1 s
Wall time: 35.2 s
type(y_hat)= <class 'torch.Tensor'> torch.Size([256, 120])
type(pred)= <class 'torch.Tensor'> torch.Size([256, 1])
type(y)= <class 'torch.Tensor'> torch.Size([256])
type(y_hat.max(1, keepdim=True))= <class 'torch.return_types.max'>
type(y_hat)= <class 'torch.Tensor'> torch.Size([256, 120])
type(pred)= <class 'torch.Tensor'> torch.Size([256, 1])
type(y)= <class 'torch.Tensor'> torch.Size([256])
type(y_hat.max(1, keepdim=True))= <class 'torch.return_types.max'>
type(y_hat)= <class 'torch.Tensor'> torch.Size([256, 120])
type(pred)= <class 'torch.Tensor'> torch.Size([256, 1])
type(y)= <class 'torch.Tensor'> torch.Size([256])
type(y_hat.max(1, keepdim=True))= <class 'torch.return_types.max'>
type(y_hat)= <class 'torch.Tensor'> torch.Size([255, 120])
type(pred)= <class 'torch.Tensor'> torch.Size([255, 1])
type(y)= <class 'torch.Tensor'> torch.Size([255])
type(y_hat.max(1, keepdim=T

我们看到只训练了9次就达到了80%的准确率，效果还是可以的。

但是每次训练都需要将一张图片在全部网络中进行计算，而且计算的结果每次都是一样的，这样浪费了很多计算的资源。
下面我们就将这些不进行反向传播或者说不更新网络权重参数层的计算结果保存下来，这样我们以后使用的时候就可以直接将这些结果输入到FC层或者以这些结果构建新的网络层，省去了计算的时间，并且这样如果只训练全连接层，CPU就可以完成了。

4.1.4 固定层的向量导出
[PyTorch论坛](https://discuss.pytorch.org/t/can-i-get-the-middle-layers-output-if-i-use-the-sequential-module/7070)中说到可以使用自己手动实现模型中的forward参数，这样看起来是很简便的，但是这样处理起来很麻烦，不建议这样使用。

这里我们就要采用PyTorch比较高级的API，hook来处理了，我们要先定义一个hook函数

In [79]:
testzwtuple=(1,)
print(testzwtuple[0])

1


In [83]:
# PyTorch前向钩子函数，绑定到模型层后，前向传播时自动触发
# 参数说明：
# - module：当前钩子绑定的模型层（如conv1、fc、layer4等）
# - input：该层的输入，tuple类型（即使只有1个输入，也会封装为tuple）
# - output：该层的输出，tensor/tuple类型（你代码中未用到
in_list= [] # 存储所有捕获的输入特征（NumPy格式）
def hook(module, input, output):
    #input是一个tuple代表顺序代表每一个输入项，我们这里只有一项，所以直接获取
    #需要全部的参数信息可以使用这个打印
    # input.shape 报错，AttributeError: 'tuple' object has no attribute 'shape'

    # input= <class 'tuple'> (tensor([[[[3.8350e-01....., 1.8616e+00]]]], device='mps:0'),)
    # input[0].shape = torch.Size([256, 2048, 7, 7])
    # 元组末尾的 , 是 Python 中单元素元组的语法标识（(x,) 表示只有一个元素 x 的元组）
    #  张量的维度结构（以 Conv2d 层为例）：
    # - 第 1 层维度：批次维度（batch_size）；
    # - 第 2 层维度：通道维度（如 3 通道 RGB）；
    # - 第 3/4 层维度：图像的高 / 宽（如 224×224）；
    # - 数值是归一化后的像素值（科学计数法，如 3.8350e-01=0.3835）；
    # 3.8350e-01 = 3.8350 × 10^(-1)  说明 a = 3.8350（系数）  e = ×10 的... 次方 b = -01（指数，即 -1）
    # print('input=',type(input),input[0].shape)

    # output= <class 'torch.Tensor'> tensor([[[[0.6541]],... [[0.3019]]]], device='mps:0')
    # print('output=',type(output),output)
    # for val in input:
    #    print("input val:",val)
    # PyTorch 张量的 size(dim) 方法用于获取指定维度的长度，dim=0 对应张量的第一个维度（批次维
    # input[0].size(0)=256
    for i in range(input[0].size(0)):
        print('input[0][i] = ',type(input[0][i]),input[0][i].shape)
        # input[0][i]	[2048, 7, 7]   第 i 个样本的特征张量
        # .cpu() 方法会创建一个新的张量副本，将原张量的数据从 GPU/MPS 设备转移到 CPU 上，原张量仍保留在原设备，新张量在 CPU 上
        in_list.append(input[0][i].cpu().numpy())

在相应的层注册hook函数，保证函数能够正常工作，我们这里直接hook 全连接层前面的pool层，获取pool层的输入数据，这样会获得更多的特征

In [84]:
model_ft.avgpool.register_forward_hook(hook)

开始获取输出，这里我们因为不需要反向传播，所以直接可以使用no_grad嵌套

In [85]:
%%time
with torch.no_grad():
    for batch_idx, data in enumerate(image_dataloader["train"]):
        x,y= data
        x=x.to(DEVICE)
        y=y.to(DEVICE)
        y_hat = model_ft(x)

input[0][i] <class 'torch.Tensor'> torch.Size([2048, 7, 7]) tensor([[[0.0000, 0.0924, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.3752, 0.0000,  ..., 0.0000, 0.1964, 0.1733],
         [0.1507, 0.9957, 0.5262,  ..., 0.7273, 1.0608, 0.2772],
         ...,
         [0.5692, 1.3567, 1.9137,  ..., 2.1861, 1.6013, 0.4752],
         [0.0705, 1.4471, 1.4504,  ..., 3.1341, 1.8363, 0.3814],
         [0.5667, 0.8392, 1.1420,  ..., 3.4413, 2.1444, 0.7045]],

        [[1.6907, 0.8576, 0.2704,  ..., 0.0000, 0.0000, 0.0000],
         [2.0080, 1.0269, 0.2813,  ..., 0.0000, 0.0000, 0.0000],
         [1.7550, 0.8098, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         ...,
         [0.7231, 0.1258, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.1132, 0.3843,  ..., 0.1892, 0.0000, 0.0000],
         [0.0000, 0.3622, 0.6626,  ..., 1.3258, 0.1253, 0.0000]],

        [[0.6267, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.5099, 0.1407, 0.0000,  ..., 0.0000, 0.0000, 0.0000]

KeyboardInterrupt: 

In [ ]:
# 转换为 NumPy 数组
features=np.array(in_list)
# 保存为.npy格式的文件（文件名features.npy），方便后续离线分析、可视化或复用这些特征。
np.save("features",features)

In [86]:
# 加载features.npy文件（文件需在当前代码运行目录下）
features_loaded = np.load("features.npy")
print("特征数组形状：", features_loaded.shape)  # 输出 (9199, 2048, 7, 7)
print("数据类型：", features_loaded.dtype)    # 输出 float32
print("样本总数：", features_loaded.shape[0]) # 输出 9199 样本总量

特征数组形状： (9199, 2048, 7, 7)
数据类型： float32
样本总数： 9199


这样再训练时我们只需将这个数组读出来，然后可以直接使用这个数组再输入到linear或者我们前面讲到的sigmod层就可以了。

我们在这里在pool层前获取了更多的特征，可以将这些特征使用更高级的分类器，例如SVM，树型的分类器进行分类。

以上就是针对于计算机视觉方向的微调介绍，对于NLP方向来讲fastai的创始人Jeremy 在今年出发布了ULMFiT可以作为很好的参考
具体请看这两个链接：

[fast.ai官方blog](https://nlp.fast.ai/)，[原论文：Universal Language Model Fine-tuning for Text Classification](https://arxiv.org/abs/1801.06146)